In [28]:
import os, time, json, math, logging
import pandas as pd
from openai import OpenAI
import json

import os
api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    raise SystemExit("OPENAI_API_KEY not set")

client = OpenAI(api_key=api_key)

In [2]:
print("API KEY IS ",api_key[:20],"...." )

API KEY IS  sk-proj-X7OnFk3EPWpE ....


In [3]:
df_indiv = pd.read_csv("../cleanData/clean_contributionsBYindividuals.csv", dtype=str)

In [4]:
df_indiv.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 158527 entries, 0 to 158526
Data columns (total 19 columns):
 #   Column                       Non-Null Count   Dtype 
---  ------                       --------------   ----- 
 0   contributor_name             158527 non-null  object
 1   contributor_first_name       158486 non-null  object
 2   contributor_last_name        158486 non-null  object
 3   contributor_street_1         158527 non-null  object
 4   contributor_street_2         24161 non-null   object
 5   contributor_city             158527 non-null  object
 6   contributor_state            158527 non-null  object
 7   contributor_zip              158527 non-null  object
 8   contributor_employer         82236 non-null   object
 9   contributor_occupation       82482 non-null   object
 10  committee_id                 158527 non-null  object
 11  contribution_receipt_date    158527 non-null  object
 12  contribution_receipt_amount  158527 non-null  object
 13  sub_id        

In [5]:
print("\nMissing values per column in indiv datase:")
print(df_indiv.isna().sum() + (df_indiv.eq('').sum())) #  # counts both NaN and empty strings


Missing values per column in indiv datase:
contributor_name                    0
contributor_first_name             41
contributor_last_name              41
contributor_street_1                0
contributor_street_2           134366
contributor_city                    0
contributor_state                   0
contributor_zip                     0
contributor_employer            76291
contributor_occupation          76045
committee_id                        0
contribution_receipt_date           0
contribution_receipt_amount         0
sub_id                              0
transaction_id                      0
year_cycle                          0
lat                                 0
lng                                 0
contributor_year                    0
dtype: int64


In [6]:
df_indiv

,contributor_name,contributor_first_name,contributor_last_name,contributor_street_1,contributor_street_2,contributor_city,contributor_state,contributor_zip,contributor_employer,contributor_occupation,committee_id,contribution_receipt_date,contribution_receipt_amount,sub_id,transaction_id,year_cycle,lat,lng,contributor_year
0,"BRONER, BLUMA",BLUMA,BRONER,2928 W ESTES AVE,NaN,CHICAGO,IL,60645,CIBC BANK USA,MANAGING DIRECTOR,C00797670,2024-03-27,150.0,4060420241953437304,A0559CC31752B492B9C4,2024,42.010456,-87.7032521,2024
1,"BRONER, BLUMA",BLUMA,BRONER,2928 W ESTES AVE,NaN,CHICAGO,IL,60645,CIBC BANK USA,MANAGING DIRECTOR,C00797670,2024-03-27,150.0,4060420241953436803,AF7643588B8CD4A91945,2024,42.010456,-87.7032521,2024
2,"SHAPIRO, DEBRA E.",DEBRA E.,SHAPIRO,"529 S FLAGLER DRIVE, 21EF",NaN,WPB,FL,33401,RETIRED,RETIRED,C00797670,2023-03-15,3000.0,4042820231745857981,A75C5E9927B57496F94B,2024,26.7089,-80.0507,2023
3,"GILLIBRAND FOR SENATE, KIRSTEN SEN.",KIRSTEN,GILLIBRAND FOR SENATE,PO BOX 150516,NaN,BROOKLYN,NY,11215,NaN,NaN,C00797670,2023-10-11,50.0,4112220231811446131,A028CAE2F953148C18E6,2024,40.6526006,-73.9497211,2023
4,"DON BACON FOR CONGRESS, DON REP.",DON,DON BACON FOR CONGRESS,P.O. BOX 391368,NaN,OMAHA,NE,68139,NaN,NaN,C00797670,2023-12-13,50.0,4020920241823946214,A1AD84CDB20F540A5885,2024,41.2587459,-95.9383758,2023
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
158522,"DUNBAR, DEAN",DEAN,DUNBAR,711 AMSTERDAM AVE,APT 27M,NEW YORK,NY,10025,SELF,REAL ESTATE,C00797670,2022-05-09,250.0,4122920221645231724,AC8622222732E4002B44,2022,40.79289,-73.97093,2022
158523,"MESSER, DAVID",DAVID,MESSER,6501 RED HOOK PLZ,STE 201,ST THOMAS,VI,00802,CRESSON MANAGEMENT,MANAGING MEMBER,C00797670,2024-11-01,40000.0,4011420251130091378,AA93FC87F07954443ABC,2024,18.3293,-64.8535,2024
158524,"MESSER, DAVID",DAVID,MESSER,6501 RED HOOK PLZ,STE 201,ST THOMAS,VI,00802,CRESSON MANAGEMENT,MANAGING MEMBER,C00797670,2024-11-01,40000.0,4011420251130091377,A7AE04608751B474FA8A,2024,18.3293,-64.8535,2024
158525,"MESSER, DAVID",DAVID,MESSER,6501 RED HOOK PLZ,STE 201,ST THOMAS,VI,00802,CRESSON MANAGEMENT,MANAGING MEMBER,C00797670,2024-11-01,51700.0,4011420251130091376,ABF03FF35477744FE95B,2024,18.3293,-64.8535,2024


In [7]:
row = df_indiv.iloc[0]


In [8]:
row

contributor_name                      BRONER, BLUMA
contributor_first_name                        BLUMA
contributor_last_name                        BRONER
contributor_street_1               2928 W ESTES AVE
contributor_street_2                            NaN
contributor_city                            CHICAGO
contributor_state                                IL
contributor_zip                               60645
contributor_employer                  CIBC BANK USA
contributor_occupation            MANAGING DIRECTOR
committee_id                              C00797670
contribution_receipt_date                2024-03-27
contribution_receipt_amount                   150.0
sub_id                          4060420241953437304
transaction_id                 A0559CC31752B492B9C4
year_cycle                                     2024
lat                                       42.010456
lng                                     -87.7032521
contributor_year                               2024
Name: 0, dty

In [32]:
def build_prompt(row):
    # Use the prompt you wrote but we add a unique id to help trace
    prompt = f"""

        You are tasked with enriching missing contributor data from FEC donation records.
        UniqueID: {row.get('transaction_id','')}

        Given the following contributor information:

        First Name: {row['contributor_first_name']}
        Last Name: {row['contributor_last_name']}
        Home Address: {row['contributor_street_1']} {row['contributor_street_2']}, {row['contributor_city']}, {row['contributor_state']} {row['contributor_zip']}
        Employer (if available): {row['contributor_employer']}
        Occupation (if available): {row['contributor_occupation']}

        Instructions:
        - Search reliable public sources (LinkedIn, company sites, news, official registries, FEC records).
        - Use only reliable public sources if you have them. If you do not have exact info, provide the best reasonable answer.
        - ALWAYS return EXACTLY one JSON object (no extra explanation, no markdown)
        - Cross-check and return the MOST likely correct values.
        - Return keys: "employer", "occupation", "contributor_business_address".
        - If missing or uncertain, use the closest reasonable answer (e.g., "Self-Employed"  -  "retired").


        JSON Schema:
        {{
          "employer": "...",
          "occupation": "...",
          "contributor_business_address": "..."
        }}
"""
    return prompt


In [33]:
def extract_json_from_response(response):
    # robust extraction: 
    text = getattr(response, "output_text", None)
    if not text:
        try:
            # Many SDK responses have response.output which can be complex
            out = getattr(response, "output", None)
            if isinstance(out, list):
                parts = []
                for item in out:
                    if isinstance(item, dict):
                        for c in item.get("content", []):
                            if isinstance(c, dict) and "text" in c:
                                parts.append(c["text"])
                            elif isinstance(c, str):
                                parts.append(c)
                    elif isinstance(item, str):
                        parts.append(item)
                text = "\n".join(parts)
        except Exception:
            text = None

    if not text:
        return None

    # strip code fences 
    text = text.strip()
    if text.startswith("```"):
        # remove fenced block
        try:
            text = text.split("\n", 1)[1].rsplit("```", 1)[0].strip()
        except Exception:
            pass

    # find first { and last } to be more robust
    start = text.find("{")
    end = text.rfind("}")
    if start != -1 and end != -1 and end > start:
        text = text[start:end+1]

    try:
        return json.loads(text)
    except Exception:
        # final fallback: return raw text in a wrapper for inspection
        return {"_raw": text}

In [41]:
def call_model(prompt, max_retries=3):
    for attempt in range(max_retries):
        try:
            resp = client.responses.create(
                model="gpt-5-mini",
                input=[{"role": "user", "content": prompt}]
            )
            return extract_json_from_response(resp)
        except Exception as e:
            wait = 2 ** attempt
            logging.warning(f"Model call failed (attempt {attempt+1}): {e}; retrying in {wait}s")
            time.sleep(wait)
    return None


## real work 

In [10]:
#  df with rows where both contributor_employer and contributor_occupation are missing
df_missing = df_indiv[
    (df_indiv['contributor_employer'].isna() | df_indiv['contributor_employer'].eq('')) &
    (df_indiv['contributor_occupation'].isna() | df_indiv['contributor_occupation'].eq(''))
]

In [42]:
# Extract unique contributors based on contributor_name
df_unique_contributors = df_missing.drop_duplicates(subset=['contributor_name'], keep='first')

# Optionally, select only relevant columns for contributors
df_unique_contributors = df_unique_contributors[[
    'contributor_name', 
    'contributor_first_name', 
    'contributor_last_name', 
    'contributor_street_1', 
    'contributor_street_2', 
    'contributor_city', 
    'contributor_state', 
    'contributor_zip', 
    'contributor_employer',
    'contributor_occupation',
]]  

# Verify the result
print(f"Number of unique contributors: {len(df_unique_contributors)}")
print("\nUnique contributors DataFrame:")
print(df_unique_contributors.head())

Number of unique contributors: 653

Unique contributors DataFrame:
                        contributor_name contributor_first_name  \
3    GILLIBRAND FOR SENATE, KIRSTEN SEN.                KIRSTEN   
4       DON BACON FOR CONGRESS, DON REP.                    DON   
21                 DON DAVIS FOR NC, DON                    DON   
62               TEAM HAGERTY, BILL SEN.                   BILL   
140                              CITYPAC                    NaN   

      contributor_last_name contributor_street_1 contributor_street_2  \
3     GILLIBRAND FOR SENATE        PO BOX 150516                  NaN   
4    DON BACON FOR CONGRESS      P.O. BOX 391368                  NaN   
21         DON DAVIS FOR NC           PO BOX 511                  NaN   
62             TEAM HAGERTY         PO BOX 50430                  NaN   
140                     NaN       745 MARION AVE                  NaN   

    contributor_city contributor_state contributor_zip contributor_employer  \
3           

In [65]:
df_unique_contributors.info()

<class 'pandas.core.frame.DataFrame'>
Index: 653 entries, 3 to 158256
Data columns (total 11 columns):
 #   Column                        Non-Null Count  Dtype 
---  ------                        --------------  ----- 
 0   contributor_name              653 non-null    object
 1   contributor_first_name        652 non-null    object
 2   contributor_last_name         652 non-null    object
 3   contributor_street_1          653 non-null    object
 4   contributor_street_2          107 non-null    object
 5   contributor_city              653 non-null    object
 6   contributor_state             653 non-null    object
 7   contributor_zip               653 non-null    object
 8   contributor_business_address  5 non-null      object
 9   contributor_employer          5 non-null      object
 10  contributor_occupation        5 non-null      object
dtypes: object(11)
memory usage: 77.4+ KB


In [53]:
print("\nMissing values per column in indiv datase:")
print(df_unique_contributors.isna().sum() + (df_unique_contributors.eq('').sum())) #  # counts both NaN and empty strings


Missing values per column in indiv datase:
contributor_name                  0
contributor_first_name            1
contributor_last_name             1
contributor_street_1              0
contributor_street_2            546
contributor_city                  0
contributor_state                 0
contributor_zip                   0
contributor_business_address    653
contributor_employer            653
contributor_occupation          653
dtype: int64


### add columns for bnusiness adress 

In [64]:
df_unique_contributors["contributor_business_address"] = (
    df_unique_contributors["contributor_business_address"].astype("object")
)

In [ ]:
import numpy as np
insert_position = df_unique_contributors.columns.get_loc('contributor_zip') + 1

# Insert the new column with NaN or empty string as default values
df_unique_contributors.insert(insert_position, 'contributor_business_address', np.nan)

df_unique_contributors["contributor_business_address"] = (
    df_unique_contributors["contributor_business_address"].astype("object")
)

print("Updated DataFrame columns:")
print(df_unique_contributors.columns.tolist())
print("\nFirst few rows of updated DataFrame:")
print(df_unique_contributors.head())

Updated DataFrame columns:
['contributor_name', 'contributor_first_name', 'contributor_last_name', 'contributor_street_1', 'contributor_street_2', 'contributor_city', 'contributor_state', 'contributor_zip', 'contributor_business_address', 'contributor_employer', 'contributor_occupation']

First few rows of updated DataFrame:
                        contributor_name contributor_first_name  \
3    GILLIBRAND FOR SENATE, KIRSTEN SEN.                KIRSTEN   
4       DON BACON FOR CONGRESS, DON REP.                    DON   
21                 DON DAVIS FOR NC, DON                    DON   
62               TEAM HAGERTY, BILL SEN.                   BILL   
140                              CITYPAC                    NaN   

      contributor_last_name contributor_street_1 contributor_street_2  \
3     GILLIBRAND FOR SENATE        PO BOX 150516                  NaN   
4    DON BACON FOR CONGRESS      P.O. BOX 391368                  NaN   
21         DON DAVIS FOR NC           PO BOX 511   

In [66]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.StreamHandler()
    ]
)

total_rows = len(df_unique_contributors)
processed = 0

for idx, row in df_unique_contributors.iterrows():
    if pd.isna(row["contributor_employer"]) or pd.isna(row["contributor_occupation"]) or pd.isna(row["contributor_business_address"]):
        processed += 1
        percent_done = (processed / total_rows) * 100

        logging.info(
            f"Processing row {idx} "
            f"({processed}/{total_rows} → {percent_done:.2f}% done) | "
            f"Contributor: {row['contributor_first_name']} {row['contributor_last_name']}"
        )

        prompt = build_prompt(row)
        result = call_model(prompt)

        if result is None:
            logging.warning(f"Row {idx}: Model returned None")
            continue  

        try:
            df_unique_contributors.at[idx, "contributor_employer"] = result.get("employer", "")
            df_unique_contributors.at[idx, "contributor_occupation"] = result.get("occupation", "")
            df_unique_contributors.at[idx, "contributor_business_address"] = result.get("contributor_business_address", "")

            logging.info(
                f"Row {idx} updated → Employer: {result.get('employer','')}, "
                f"Occupation: {result.get('occupation','')}, "
                f"Business Address: {result.get('contributor_business_address','')}"
            )

        except Exception as e:
            logging.error(f"Row {idx}: Failed to update due to error: {e}")

        # small delay to respect API rate limits
        time.sleep(0.5)

# Save results
df_unique_contributors.to_csv("enriched_contributors.csv", index=False)
logging.info("✅ Data enrichment complete. Saved to enriched_contributors.csv")


In [67]:
df_unique_contributors.info()

<class 'pandas.core.frame.DataFrame'>
Index: 653 entries, 3 to 158256
Data columns (total 11 columns):
 #   Column                        Non-Null Count  Dtype 
---  ------                        --------------  ----- 
 0   contributor_name              653 non-null    object
 1   contributor_first_name        652 non-null    object
 2   contributor_last_name         652 non-null    object
 3   contributor_street_1          653 non-null    object
 4   contributor_street_2          107 non-null    object
 5   contributor_city              653 non-null    object
 6   contributor_state             653 non-null    object
 7   contributor_zip               653 non-null    object
 8   contributor_business_address  653 non-null    object
 9   contributor_employer          653 non-null    object
 10  contributor_occupation        653 non-null    object
dtypes: object(11)
memory usage: 77.4+ KB


In [70]:
df_merged = df_indiv.merge(
    df_unique_contributors[['contributor_name', 'contributor_employer', 'contributor_occupation', 'contributor_business_address']],
    on='contributor_name',
    how='left',
    suffixes=('', '_updated')
)

# Update contributor_employer and contributor_occupation with values from df_unique_contributors where available
df_merged['contributor_employer'] = df_merged['contributor_employer_updated'].combine_first(df_merged['contributor_employer'])
df_merged['contributor_occupation'] = df_merged['contributor_occupation_updated'].combine_first(df_merged['contributor_occupation'])

# Add contributor_business_address (will be NaN for non-matching rows)
# It's already included from the merge, so no additional step needed unless you want to rename it

# Drop the temporary updated columns
df_merged = df_merged.drop(columns=['contributor_employer_updated', 'contributor_occupation_updated'])

# Verify the result
print(df_merged.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 158527 entries, 0 to 158526
Data columns (total 20 columns):
 #   Column                        Non-Null Count   Dtype 
---  ------                        --------------   ----- 
 0   contributor_name              158527 non-null  object
 1   contributor_first_name        158486 non-null  object
 2   contributor_last_name         158486 non-null  object
 3   contributor_street_1          158527 non-null  object
 4   contributor_street_2          24161 non-null   object
 5   contributor_city              158527 non-null  object
 6   contributor_state             158527 non-null  object
 7   contributor_zip               158527 non-null  object
 8   contributor_employer          158261 non-null  object
 9   contributor_occupation        158509 non-null  object
 10  committee_id                  158527 non-null  object
 11  contribution_receipt_date     158527 non-null  object
 12  contribution_receipt_amount   158527 non-null  object
 13 

In [71]:
print("\nMissing values per column in indiv datase:")
print(df_merged.isna().sum() + (df_merged.eq('').sum())) #  # counts both NaN and empty strings


Missing values per column in indiv datase:
contributor_name                     0
contributor_first_name              41
contributor_last_name               41
contributor_street_1                 0
contributor_street_2            134366
contributor_city                     0
contributor_state                    0
contributor_zip                      0
contributor_employer               266
contributor_occupation              18
committee_id                         0
contribution_receipt_date            0
contribution_receipt_amount          0
sub_id                               0
transaction_id                       0
year_cycle                           0
lat                                  0
lng                                  0
contributor_year                     0
contributor_business_address     82098
dtype: int64


In [73]:
df_new_enhanced = pd.read_csv("enriched_contributors_indiv.csv", dtype=str)

In [75]:
print("\nMissing values per column in indiv datase:")
print(df_new_enhanced.isna().sum() + (df_new_enhanced.eq('').sum())) #  # counts both NaN and empty strings


Missing values per column in indiv datase:
contributor_name                     0
contributor_first_name              41
contributor_last_name               41
contributor_street_1                 0
contributor_street_2            134347
contributor_city                     0
contributor_state                    0
contributor_zip                      0
contributor_employer               228
contributor_occupation               0
committee_id                         0
contribution_receipt_date            0
contribution_receipt_amount          0
sub_id                               0
transaction_id                       0
year_cycle                           0
lat                                  0
lng                                  0
contributor_year                     0
contributor_business_address     81888
dtype: int64


In [83]:
def build_prompt_2(row):
    # Use the prompt you wrote but we add a unique id to help trace
    prompt = f"""

        You are tasked with enriching missing contributor data from FEC donation records.
        UniqueID: {row.get('transaction_id','')}

        Given the following contributor information:

        First Name: {row['contributor_first_name']}
        Last Name: {row['contributor_last_name']}
        Home Address: {row['contributor_street_1']} {row['contributor_street_2']}, {row['contributor_city']}, {row['contributor_state']} {row['contributor_zip']}
        Employer (if available): {row['contributor_employer']}
        Occupation (if available): {row['contributor_occupation']}

        Instructions:
        - Search reliable public sources (LinkedIn, company sites, news, official registries, FEC records).
        - ALWAYS return EXACTLY one JSON object (no extra explanation, no markdown)
        - Cross-check and return the MOST likely correct values.
        - Return keys: "contributor_business_address".
        - If missing or uncertain, use the closest reasonable answer or write not available.

        JSON Schema:
        {{
          "contributor_business_address": "..."
        }}
"""
    return prompt


In [80]:
test_row = df_new_enhanced[df_new_enhanced['contributor_business_address'].isna() |
                           (df_new_enhanced['contributor_business_address'].str.strip() == '')].iloc[0]



test_row


contributor_name                       BRONER, BLUMA
contributor_first_name                         BLUMA
contributor_last_name                         BRONER
contributor_street_1                2928 W ESTES AVE
contributor_street_2                             NaN
contributor_city                             CHICAGO
contributor_state                                 IL
contributor_zip                                60645
contributor_employer                   CIBC BANK USA
contributor_occupation             MANAGING DIRECTOR
committee_id                               C00797670
contribution_receipt_date                  3/27/2024
contribution_receipt_amount                      150
sub_id                                   4.06042E+18
transaction_id                  A0559CC31752B492B9C4
year_cycle                                      2024
lat                                        42.010456
lng                                      -87.7032521
contributor_year                              

In [87]:
missing_address_df = df_new_enhanced[pd.isna(df_new_enhanced["contributor_business_address"])]


In [91]:
unique_missing_df = missing_address_df.drop_duplicates(subset=["contributor_first_name", "contributor_last_name"])


In [92]:
unique_missing_df

,contributor_name,contributor_first_name,contributor_last_name,contributor_street_1,contributor_street_2,contributor_city,contributor_state,contributor_zip,contributor_employer,contributor_occupation,committee_id,contribution_receipt_date,contribution_receipt_amount,sub_id,transaction_id,year_cycle,lat,lng,contributor_year,contributor_business_address
0,"BRONER, BLUMA",BLUMA,BRONER,2928 W ESTES AVE,NaN,CHICAGO,IL,60645,CIBC BANK USA,MANAGING DIRECTOR,C00797670,3/27/2024,150,4.06042E+18,A0559CC31752B492B9C4,2024,42.010456,-87.7032521,2024,NaN
2,"SHAPIRO, DEBRA E.",DEBRA E.,SHAPIRO,"529 S FLAGLER DRIVE, 21EF",NaN,WPB,FL,33401,RETIRED,RETIRED,C00797670,3/15/2023,3000,4.04282E+18,A75C5E9927B57496F94B,2024,26.7089,-80.0507,2023,NaN
10,"ROMBOM, HOWARD",HOWARD,ROMBOM,310 E SHORE RD,STE 100,GREAT NECK,NY,11023,SELF,CONSULTANT,C00797670,8/12/2024,500,4.09232E+18,A448E8934B7C140FABA2,2024,40.8003181,-73.7137501,2024,NaN
33,"BERMAN, NELSON",NELSON,BERMAN,529 KINGS DR,NaN,CHERRY HILL,NJ,8003,SELF,SALES,C00797670,11/28/2023,180,4.12212E+18,A3CAF372530BE4F3BA13,2024,39.8703225,-74.9746697,2023,NaN
42,"DOW, MELVIN",MELVIN,DOW,11107 HEDWIG LANE,NaN,HOUSTON,TX,77024,RETIRED,RETIRED,C00797670,3/31/2023,1000,4.04282E+18,A6EDBBAE78700485384D,2024,29.7589382,-95.3676974,2023,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
158489,"MESSER, DAVID",DAVID,MESSER,6501 RED HOOK PLZ,STE 201,ST THOMAS,VI,802,CRESSON MANAGEMENT,MANAGING MEMBER,C00797670,11/1/2024,20000,4.01142E+18,A34B3B7CE3B014135839,2024,18.3293,-64.8535,2024,NaN
158492,"ROSENBAUM, STEVE",STEVE,ROSENBAUM,2208 SHEFFLIN CT,NaN,BALTIMORE,MD,21209,QUEST,REAL ESTATE,C00797670,5/31/2024,15,4.06242E+18,A7DDEAC3114894671A4A,2024,39.28471,-76.69073,2024,NaN
158493,"ROSENFELD, MELODIE",MELODIE,ROSENFELD,13 HISH ST.,NaN,REHOVOT,CA,95066,TALPIOT COLLEGE,LECTURER,C00797670,5/31/2024,100,4.06242E+18,A3106ECB4D0334829A60,2024,31.894756,-76.69073,2024,NaN
158495,"DUNBAR, DEAN",DEAN,DUNBAR,711 AMSTERDAM AVE,APT 27M,NEW YORK,NY,10025,SELF,REAL ESTATE,C00797670,5/9/2022,250,4.12292E+18,AC8622222732E4002B44,2022,40.79289,-73.97093,2022,NaN


In [93]:
# --- Configure logging for Jupyter ---
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[logging.StreamHandler()]  # prints to notebook output
)

processed = 0
total_rows = len(unique_missing_df)

for idx, row in unique_missing_df.iterrows():
    if pd.isna(row["contributor_employer"]) or pd.isna(row["contributor_occupation"]) or pd.isna(row["contributor_business_address"]):
        processed += 1
        percent_done = (processed / total_rows) * 100

        logging.info(
            f"Processing row {idx} "
            f"({processed}/{total_rows} → {percent_done:.2f}% done) | "
            f"Contributor: {row['contributor_first_name']} {row['contributor_last_name']}"
        )

        # Make sure you use the current row, not 'test_row'
        prompt = build_prompt_2(row)  
        result = call_model(prompt)
        if result is None:
            logging.warning(f"Row {idx}: Model returned None")
            continue  

        try:
            unique_missing_df.at[idx, "contributor_business_address"] = result.get("contributor_business_address", "")

            logging.info(
                f"Row {idx} updated → Employer: {result.get('employer','')}, "
                f"Occupation: {result.get('occupation','')}, "
                f"Business Address: {result.get('contributor_business_address','')}"
            )

        except Exception as e:
            logging.error(f"Row {idx}: Failed to update due to error: {e}")

        # small delay to respect API rate limits
        time.sleep(0.5)


# Save results
unique_missing_df.to_csv("enriched_contributorsWITH BUSINESS.csv", index=False)
logging.info("✅ Data enrichment complete. Saved to enriched_contributors.csv")


KeyboardInterrupt: 

In [94]:
unique_missing_df

,contributor_name,contributor_first_name,contributor_last_name,contributor_street_1,contributor_street_2,contributor_city,contributor_state,contributor_zip,contributor_employer,contributor_occupation,committee_id,contribution_receipt_date,contribution_receipt_amount,sub_id,transaction_id,year_cycle,lat,lng,contributor_year,contributor_business_address
0,"BRONER, BLUMA",BLUMA,BRONER,2928 W ESTES AVE,NaN,CHICAGO,IL,60645,CIBC BANK USA,MANAGING DIRECTOR,C00797670,3/27/2024,150,4.06042E+18,A0559CC31752B492B9C4,2024,42.010456,-87.7032521,2024,"CIBC Bank USA, 120 S LaSalle St, Chicago, IL 6..."
2,"SHAPIRO, DEBRA E.",DEBRA E.,SHAPIRO,"529 S FLAGLER DRIVE, 21EF",NaN,WPB,FL,33401,RETIRED,RETIRED,C00797670,3/15/2023,3000,4.04282E+18,A75C5E9927B57496F94B,2024,26.7089,-80.0507,2023,not available
10,"ROMBOM, HOWARD",HOWARD,ROMBOM,310 E SHORE RD,STE 100,GREAT NECK,NY,11023,SELF,CONSULTANT,C00797670,8/12/2024,500,4.09232E+18,A448E8934B7C140FABA2,2024,40.8003181,-73.7137501,2024,"310 E SHORE RD STE 100, GREAT NECK, NY 11023"
33,"BERMAN, NELSON",NELSON,BERMAN,529 KINGS DR,NaN,CHERRY HILL,NJ,8003,SELF,SALES,C00797670,11/28/2023,180,4.12212E+18,A3CAF372530BE4F3BA13,2024,39.8703225,-74.9746697,2023,"529 KINGS DR, CHERRY HILL, NJ 08003"
42,"DOW, MELVIN",MELVIN,DOW,11107 HEDWIG LANE,NaN,HOUSTON,TX,77024,RETIRED,RETIRED,C00797670,3/31/2023,1000,4.04282E+18,A6EDBBAE78700485384D,2024,29.7589382,-95.3676974,2023,not available
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
158489,"MESSER, DAVID",DAVID,MESSER,6501 RED HOOK PLZ,STE 201,ST THOMAS,VI,802,CRESSON MANAGEMENT,MANAGING MEMBER,C00797670,11/1/2024,20000,4.01142E+18,A34B3B7CE3B014135839,2024,18.3293,-64.8535,2024,NaN
158492,"ROSENBAUM, STEVE",STEVE,ROSENBAUM,2208 SHEFFLIN CT,NaN,BALTIMORE,MD,21209,QUEST,REAL ESTATE,C00797670,5/31/2024,15,4.06242E+18,A7DDEAC3114894671A4A,2024,39.28471,-76.69073,2024,NaN
158493,"ROSENFELD, MELODIE",MELODIE,ROSENFELD,13 HISH ST.,NaN,REHOVOT,CA,95066,TALPIOT COLLEGE,LECTURER,C00797670,5/31/2024,100,4.06242E+18,A3106ECB4D0334829A60,2024,31.894756,-76.69073,2024,NaN
158495,"DUNBAR, DEAN",DEAN,DUNBAR,711 AMSTERDAM AVE,APT 27M,NEW YORK,NY,10025,SELF,REAL ESTATE,C00797670,5/9/2022,250,4.12292E+18,AC8622222732E4002B44,2022,40.79289,-73.97093,2022,NaN


In [95]:
print("\nMissing values per column in indiv datase:")
print(unique_missing_df.isna().sum() + (unique_missing_df.eq('').sum())) #  # counts both NaN and empty strings


Missing values per column in indiv datase:
contributor_name                    0
contributor_first_name              0
contributor_last_name               0
contributor_street_1                0
contributor_street_2            13175
contributor_city                    0
contributor_state                   0
contributor_zip                     0
contributor_employer               35
contributor_occupation              0
committee_id                        0
contribution_receipt_date           0
contribution_receipt_amount         0
sub_id                              0
transaction_id                      0
year_cycle                          0
lat                                 0
lng                                 0
contributor_year                    0
contributor_business_address    10046
dtype: int64


In [96]:
unique_missing_df.to_csv("enriched_contributorsWITH BUSINESS.csv", index=False)


In [97]:
address_map = unique_missing_df.set_index("contributor_name")["contributor_business_address"]

In [98]:
address_map

contributor_name
BRONER, BLUMA         CIBC Bank USA, 120 S LaSalle St, Chicago, IL 6...
SHAPIRO, DEBRA E.                                         not available
ROMBOM, HOWARD             310 E SHORE RD STE 100, GREAT NECK, NY 11023
BERMAN, NELSON                      529 KINGS DR, CHERRY HILL, NJ 08003
DOW, MELVIN                                               not available
                                            ...                        
MESSER, DAVID                                                       NaN
ROSENBAUM, STEVE                                                    NaN
ROSENFELD, MELODIE                                                  NaN
DUNBAR, DEAN                                                        NaN
DANZIG, GABRIEL                                                     NaN
Name: contributor_business_address, Length: 15685, dtype: object

In [99]:
len(address_map)

15685

In [100]:
df_new_enhanced["contributor_business_address"] = (
    df_new_enhanced["contributor_business_address"]
    .fillna(df_new_enhanced["contributor_name"].map(address_map))
)


In [101]:
df_new_enhanced

,contributor_name,contributor_first_name,contributor_last_name,contributor_street_1,contributor_street_2,contributor_city,contributor_state,contributor_zip,contributor_employer,contributor_occupation,committee_id,contribution_receipt_date,contribution_receipt_amount,sub_id,transaction_id,year_cycle,lat,lng,contributor_year,contributor_business_address
0,"BRONER, BLUMA",BLUMA,BRONER,2928 W ESTES AVE,NaN,CHICAGO,IL,60645,CIBC BANK USA,MANAGING DIRECTOR,C00797670,3/27/2024,150,4.06042E+18,A0559CC31752B492B9C4,2024,42.010456,-87.7032521,2024,"CIBC Bank USA, 120 S LaSalle St, Chicago, IL 6..."
1,"BRONER, BLUMA",BLUMA,BRONER,2928 W ESTES AVE,NaN,CHICAGO,IL,60645,CIBC BANK USA,MANAGING DIRECTOR,C00797670,3/27/2024,150,4.06042E+18,AF7643588B8CD4A91945,2024,42.010456,-87.7032521,2024,"CIBC Bank USA, 120 S LaSalle St, Chicago, IL 6..."
2,"SHAPIRO, DEBRA E.",DEBRA E.,SHAPIRO,"529 S FLAGLER DRIVE, 21EF",NaN,WPB,FL,33401,RETIRED,RETIRED,C00797670,3/15/2023,3000,4.04282E+18,A75C5E9927B57496F94B,2024,26.7089,-80.0507,2023,not available
3,"GILLIBRAND FOR SENATE, KIRSTEN SEN.",KIRSTEN,GILLIBRAND FOR SENATE,PO BOX 150516,NaN,BROOKLYN,NY,11215,Kirsten Gillibrand for Senate (campaign commit...,Campaign committee,C00797670,10/11/2023,50,4.11222E+18,A028CAE2F953148C18E6,2024,40.6526006,-73.9497211,2023,"PO Box 150516, Brooklyn, NY 11215"
4,"DON BACON FOR CONGRESS, DON REP.",DON,DON BACON FOR CONGRESS,P.O. BOX 391368,NaN,OMAHA,NE,68139,Don Bacon for Congress,Campaign Committee,C00797670,12/13/2023,50,4.02092E+18,A1AD84CDB20F540A5885,2024,41.2587459,-95.9383758,2023,"P.O. BOX 391368, Omaha, NE 68139"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
158495,"DUNBAR, DEAN",DEAN,DUNBAR,711 AMSTERDAM AVE,APT 27M,NEW YORK,NY,10025,SELF,REAL ESTATE,C00797670,5/9/2022,250,4.12292E+18,AC8622222732E4002B44,2022,40.79289,-73.97093,2022,NaN
158496,"MESSER, DAVID",DAVID,MESSER,6501 RED HOOK PLZ,STE 201,ST THOMAS,VI,802,CRESSON MANAGEMENT,MANAGING MEMBER,C00797670,11/1/2024,40000,4.01142E+18,AA93FC87F07954443ABC,2024,18.3293,-64.8535,2024,NaN
158497,"MESSER, DAVID",DAVID,MESSER,6501 RED HOOK PLZ,STE 201,ST THOMAS,VI,802,CRESSON MANAGEMENT,MANAGING MEMBER,C00797670,11/1/2024,40000,4.01142E+18,A7AE04608751B474FA8A,2024,18.3293,-64.8535,2024,NaN
158498,"MESSER, DAVID",DAVID,MESSER,6501 RED HOOK PLZ,STE 201,ST THOMAS,VI,802,CRESSON MANAGEMENT,MANAGING MEMBER,C00797670,11/1/2024,51700,4.01142E+18,ABF03FF35477744FE95B,2024,18.3293,-64.8535,2024,NaN


In [ ]:
print("\nMissing values per column in indiv datase:")
print(df_new_enhanced.isna().sum() + (df_new_enhanced.eq('').sum()))


Missing values per column in indiv datase:
contributor_name                     0
contributor_first_name              41
contributor_last_name               41
contributor_street_1                 0
contributor_street_2            134347
contributor_city                     0
contributor_state                    0
contributor_zip                      0
contributor_employer               228
contributor_occupation               0
committee_id                         0
contribution_receipt_date            0
contribution_receipt_amount          0
sub_id                               0
transaction_id                       0
year_cycle                           0
lat                                  0
lng                                  0
contributor_year                     0
contributor_business_address     31441
dtype: int64


In [104]:
df_new_enhanced.to_csv("../cleanData/clean_contrib_INDIV_enriched.csv", index=False)
